# 02 - Programacao Cientifica com Claude
> Debug, otimizacao e conversao de codigo para simulacoes numericas

**Modulo:** `10_engfis_academico` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=4096):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

## Gerando Simulacoes Monte Carlo

O metodo de Monte Carlo usa amostragem aleatoria para resolver problemas
deterministicos. Vamos pedir ao Claude para gerar simulacoes completas,
comecando por um exemplo classico: estimar o valor de pi.

In [ ]:
system_sci = '''Voce e um assistente de programacao cientifica para engenharia fisica.
Retorne APENAS codigo Python puro, sem explicacoes, sem markdown.
Use numpy para vetorizacao e matplotlib para graficos.
Inclua comentarios breves no codigo.'''

prompt_pi = '''Escreva uma simulacao Monte Carlo para estimar pi.
- Use N = 100_000 pontos aleatorios no quadrado unitario.
- Conte quantos caem dentro do circulo de raio 1.
- Plote os pontos (dentro em azul, fora em vermelho).
- Imprima a estimativa de pi e o erro relativo.'''

codigo_pi = ask(prompt_pi, system=system_sci, model=SONNET)
print(codigo_pi)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulacao Monte Carlo para estimar pi
N = 100_000
np.random.seed(42)

x = np.random.uniform(0, 1, N)
y = np.random.uniform(0, 1, N)
distancia = x**2 + y**2
dentro = distancia <= 1.0

pi_estimado = 4.0 * np.sum(dentro) / N
erro_relativo = abs(pi_estimado - np.pi) / np.pi * 100

print(f'Estimativa de pi: {pi_estimado:.6f}')
print(f'Valor real:       {np.pi:.6f}')
print(f'Erro relativo:    {erro_relativo:.4f}%')

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.scatter(x[dentro], y[dentro], s=0.1, c='blue', alpha=0.3, label='Dentro')
ax.scatter(x[~dentro], y[~dentro], s=0.1, c='red', alpha=0.3, label='Fora')
theta = np.linspace(0, np.pi/2, 100)
ax.plot(np.cos(theta), np.sin(theta), 'k-', lw=2)
ax.set_aspect('equal')
ax.set_title(f'Monte Carlo: pi ~ {pi_estimado:.4f} (N={N:,})')
ax.legend()
plt.tight_layout()
plt.show()

### Monte Carlo: Decaimento Radioativo

Agora um exemplo mais relevante para engenharia fisica: simulacao
estocastica de decaimento radioativo de uma amostra.

In [ ]:
prompt_decay = '''Escreva uma simulacao Monte Carlo de decaimento radioativo.
- Comece com N0 = 10000 atomos.
- Probabilidade de decaimento por passo: p = 0.01
- Simule 500 passos de tempo.
- Faca 5 realizacoes estocasticas e plote todas.
- Sobreponha a solucao analitica N(t) = N0 * exp(-lambda*t) onde lambda = p.
- Plote com legenda e eixos rotulados.'''

codigo_decay = ask(prompt_decay, system=system_sci, model=SONNET)
print(codigo_decay)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parametros
N0 = 10_000
p = 0.01          # probabilidade de decaimento por passo
n_passos = 500
n_realizacoes = 5
np.random.seed(7)

t = np.arange(n_passos + 1)

fig, ax = plt.subplots(figsize=(10, 5))

for i in range(n_realizacoes):
    N_atual = N0
    historico = [N_atual]
    for _ in range(n_passos):
        # Cada atomo tem probabilidade p de decair
        decaimentos = np.sum(np.random.random(N_atual) < p)
        N_atual -= decaimentos
        historico.append(N_atual)
    ax.plot(t, historico, alpha=0.6, label=f'Realizacao {i+1}')

# Solucao analitica
N_analitico = N0 * np.exp(-p * t)
ax.plot(t, N_analitico, 'k--', lw=2, label='Analitico: N0*exp(-lambda*t)')

ax.set_xlabel('Passo de tempo')
ax.set_ylabel('Numero de atomos')
ax.set_title('Decaimento Radioativo - Monte Carlo vs Analitico')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Meia-vida
meia_vida = np.log(2) / p
print(f'Meia-vida teorica: {meia_vida:.1f} passos')

---

## Debug de Codigo Numerico

Uma das aplicacoes mais uteis do Claude e encontrar bugs em codigo numerico.
Vamos enviar um solver de equacao do calor com **bugs propositais** e ver
se o Claude consegue identificar e corrigir.

In [ ]:
# Codigo COM BUGS - solver da equacao do calor 1D por diferencas finitas
codigo_bugado = '''
import numpy as np
import matplotlib.pyplot as plt

# Parametros
L = 1.0        # comprimento da barra
T_total = 0.5  # tempo total
alpha = 0.01   # difusividade termica
Nx = 50        # pontos espaciais
Nt = 1000      # passos de tempo

dx = L / Nx
dt = T_total / Nt

# BUG 1: numero de estabilidade nao verificado
# r = alpha * dt / dx^2 deve ser <= 0.5 para metodo explicito

# Condicao inicial: pulso gaussiano
x = np.linspace(0, L, Nx)  # BUG 2: deveria ser Nx+1 pontos
u = np.exp(-100 * (x - 0.5)**2)

# Condicoes de contorno: T = 0 nas extremidades
u[0] = 0.0
u[-1] = 0.0

# Time stepping
for n in range(Nt):
    u_new = u.copy()
    for i in range(Nx):  # BUG 3: range deveria ser range(1, Nx) para nao tocar bordas
        u_new[i] = u[i] + alpha * dt / dx**2 * (u[i+1] - 2*u[i] + u[i-1])  # BUG 4: i+1 estoura
    u = u_new

plt.plot(x, u)
plt.title("Equacao do Calor 1D")
plt.show()
'''

print('Codigo bugado enviado ao Claude para analise...')
print(codigo_bugado)

In [ ]:
prompt_debug = f'''Analise o codigo numerico abaixo e encontre TODOS os bugs.
Para cada bug:
1. Diga qual e o problema
2. Explique por que causa erro numerico ou runtime error
3. Mostre a correcao

Depois, mostre o codigo completo corrigido.

```python
{codigo_bugado}
```'''

analise = ask(prompt_debug, system='Voce e um especialista em metodos numericos e Python.', model=SONNET)
print(analise)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Codigo CORRIGIDO - equacao do calor 1D
L = 1.0
T_total = 0.5
alpha = 0.01
Nx = 50
Nt = 5000  # mais passos para estabilidade

dx = L / Nx
dt = T_total / Nt

# Verificacao de estabilidade (criterio CFL)
r = alpha * dt / dx**2
print(f'Numero de estabilidade r = {r:.4f}')
assert r <= 0.5, f'Instavel! r={r:.4f} > 0.5. Aumente Nt ou diminua Nx.'

# FIX: Nx+1 pontos para incluir ambas as bordas
x = np.linspace(0, L, Nx + 1)
u = np.exp(-100 * (x - 0.5)**2)
u[0] = 0.0
u[-1] = 0.0

# Guardar snapshots
snapshots = [u.copy()]
t_snap = [0.0]

for n in range(Nt):
    u_new = u.copy()
    # FIX: range(1, Nx) para nao sobrescrever condicoes de contorno
    for i in range(1, Nx):
        u_new[i] = u[i] + r * (u[i+1] - 2*u[i] + u[i-1])
    u = u_new
    # Guardar snapshots em intervalos regulares
    if (n+1) % (Nt // 5) == 0:
        snapshots.append(u.copy())
        t_snap.append((n+1) * dt)

# Plotar evolucao temporal
fig, ax = plt.subplots(figsize=(10, 5))
for s, t_val in zip(snapshots, t_snap):
    ax.plot(x, s, label=f't = {t_val:.2f}s')
ax.set_xlabel('Posicao x (m)')
ax.set_ylabel('Temperatura')
ax.set_title('Equacao do Calor 1D - Diferencas Finitas (corrigido)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Resumo dos bugs encontrados

| Bug | Problema | Impacto |
|-----|----------|---------|
| 1 | Estabilidade nao verificada (r > 0.5) | Solucao diverge |
| 2 | `Nx` pontos em vez de `Nx+1` | Grid espacial incorreto |
| 3 | Loop `range(Nx)` inclui bordas | Sobrescreve condicoes de contorno |
| 4 | `i+1` com `i=Nx-1` causa IndexError | Acesso fora do array |

---

## Otimizacao de Performance

Codigo numerico lento e comum em fisica computacional. Vamos tomar uma
integracao numerica com loops Python e pedir ao Claude para otimizar
usando vetorizacao NumPy.

In [ ]:
import numpy as np
import time

# Versao LENTA: integracao numerica com loop Python
def integrar_lento(f, a, b, N):
    """Regra do trapezio com loop Python puro."""
    h = (b - a) / N
    resultado = 0.5 * (f(a) + f(b))
    for i in range(1, N):
        x_i = a + i * h
        resultado += f(x_i)
    return resultado * h

# Funcao de teste: integral de sin(x) de 0 a pi = 2.0
f_teste = lambda x: np.sin(x)
N = 1_000_000

t0 = time.perf_counter()
res_lento = integrar_lento(f_teste, 0, np.pi, N)
t_lento = time.perf_counter() - t0

print(f'Resultado (lento): {res_lento:.10f}')
print(f'Tempo (lento):     {t_lento:.4f}s')

In [ ]:
prompt_opt = '''Tenho esta funcao de integracao numerica em Python que esta lenta:

```python
def integrar_lento(f, a, b, N):
    h = (b - a) / N
    resultado = 0.5 * (f(a) + f(b))
    for i in range(1, N):
        x_i = a + i * h
        resultado += f(x_i)
    return resultado * h
```

Otimize para maxima performance usando:
1. Vetorizacao NumPy (eliminar o loop)
2. Explique cada mudanca
Retorne apenas a funcao otimizada com comentarios.'''

resp_opt = ask(prompt_opt, system=system_sci, model=SONNET)
print(resp_opt)

In [ ]:
import numpy as np
import time

# Versao OTIMIZADA: vetorizada com NumPy
def integrar_rapido(f, a, b, N):
    """Regra do trapezio vetorizada com NumPy."""
    # Gera todos os pontos de uma vez (sem loop)
    x = np.linspace(a, b, N + 1)
    y = f(x)
    # np.trapz ja implementa a regra do trapezio
    return np.trapz(y, x)

# Versao MANUAL vetorizada (para entender o que np.trapz faz)
def integrar_vetorizado(f, a, b, N):
    """Regra do trapezio - vetorizacao manual."""
    h = (b - a) / N
    x = np.linspace(a, b, N + 1)
    y = f(x)
    # Pesos do trapezio: 0.5 nas bordas, 1.0 no interior
    return h * (0.5 * y[0] + np.sum(y[1:-1]) + 0.5 * y[-1])

f_teste = lambda x: np.sin(x)
N = 1_000_000

# Benchmark lento
t0 = time.perf_counter()
res_lento = integrar_lento(f_teste, 0, np.pi, N)
t_lento = time.perf_counter() - t0

# Benchmark rapido (np.trapz)
t0 = time.perf_counter()
res_rapido = integrar_rapido(f_teste, 0, np.pi, N)
t_rapido = time.perf_counter() - t0

# Benchmark vetorizado manual
t0 = time.perf_counter()
res_vet = integrar_vetorizado(f_teste, 0, np.pi, N)
t_vet = time.perf_counter() - t0

print(f'{"Metodo":<25} {"Resultado":<18} {"Tempo (s)":<12} {"Speedup"}')
print('-' * 70)
print(f'{"Loop Python":<25} {res_lento:<18.10f} {t_lento:<12.4f} {"1.0x"}')
print(f'{"np.trapz":<25} {res_rapido:<18.10f} {t_rapido:<12.4f} {t_lento/t_rapido:.1f}x')
print(f'{"Vetorizado manual":<25} {res_vet:<18.10f} {t_vet:<12.4f} {t_lento/t_vet:.1f}x')
print(f'\nValor exato: 2.0000000000')

### Licoes de otimizacao

| Tecnica | Quando usar |
|---------|-------------|
| Vetorizacao NumPy | Sempre que houver loops sobre arrays |
| `np.trapz`, `np.cumsum` | Funcoes built-in sao otimizadas em C |
| Evitar loops Python | Loop Python ~ 100x mais lento que NumPy vetorizado |
| Broadcasting | Operacoes entre arrays de shapes compativeis |

---

## Conversao entre Linguagens

Em engenharia fisica, muito codigo legado esta em MATLAB/Fortran.
O Claude pode converter para Python mantendo a logica matematica.

In [ ]:
codigo_matlab = '''
% Resolver ODE: dy/dt = -2*y + sin(t), y(0) = 1
% Metodo de Runge-Kutta 4a ordem

tspan = [0 10];
h = 0.01;
t = tspan(1):h:tspan(2);
N = length(t);
y = zeros(1, N);
y(1) = 1.0;  % condicao inicial

f = @(t, y) -2*y + sin(t);

for i = 1:N-1
    k1 = h * f(t(i), y(i));
    k2 = h * f(t(i) + h/2, y(i) + k1/2);
    k3 = h * f(t(i) + h/2, y(i) + k2/2);
    k4 = h * f(t(i) + h, y(i) + k3);
    y(i+1) = y(i) + (k1 + 2*k2 + 2*k3 + k4) / 6;
end

% Operacoes matriciais
A = [1 2 3; 4 5 6; 7 8 10];
b = [1; 2; 3];
x = A \\ b;  % resolver sistema linear
eigenvalues = eig(A);

figure;
plot(t, y, 'b-', 'LineWidth', 2);
xlabel('Tempo (s)');
ylabel('y(t)');
title('ODE: dy/dt = -2y + sin(t)');
grid on;
'''

prompt_conv = f'''Converta o seguinte codigo MATLAB para Python.
Use numpy e matplotlib. Explique as diferencas principais entre MATLAB e Python
em comentarios no codigo.

```matlab
{codigo_matlab}
```'''

conversao = ask(prompt_conv, system='Voce e um especialista em MATLAB e Python numerico.', model=SONNET)
print(conversao)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Converter MATLAB -> Python: ODE com RK4
# Diferenca: MATLAB indexa de 1, Python de 0
# Diferenca: MATLAB usa @(t,y) para funcoes anonimas, Python usa lambda

h = 0.01
t = np.arange(0, 10 + h, h)  # MATLAB tspan(1):h:tspan(2) -> np.arange
N = len(t)
y = np.zeros(N)
y[0] = 1.0  # Diferenca: indice 0 em vez de 1

f = lambda t, y: -2*y + np.sin(t)  # Diferenca: lambda em vez de @()

# RK4 - logica identica, apenas indices ajustados
for i in range(N - 1):  # Diferenca: range(N-1) em vez de 1:N-1
    k1 = h * f(t[i], y[i])
    k2 = h * f(t[i] + h/2, y[i] + k1/2)
    k3 = h * f(t[i] + h/2, y[i] + k2/2)
    k4 = h * f(t[i] + h, y[i] + k3)
    y[i+1] = y[i] + (k1 + 2*k2 + 2*k3 + k4) / 6

# Operacoes matriciais
A = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 10]])  # Diferenca: np.array em vez de []
b = np.array([1, 2, 3])
x = np.linalg.solve(A, b)  # Diferenca: np.linalg.solve em vez de A\b
autovalores = np.linalg.eigvals(A)  # Diferenca: np.linalg.eigvals em vez de eig()

print(f'Solucao do sistema Ax=b: {x}')
print(f'Autovalores de A: {autovalores}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t, y, 'b-', linewidth=2)
ax.set_xlabel('Tempo (s)')
ax.set_ylabel('y(t)')
ax.set_title('ODE: dy/dt = -2y + sin(t) - RK4')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Tabela de conversao MATLAB -> Python

| MATLAB | Python (NumPy) | Nota |
|--------|---------------|------|
| `A(1,1)` | `A[0,0]` | Indexacao comeca em 0 |
| `A \ b` | `np.linalg.solve(A, b)` | Sistema linear |
| `eig(A)` | `np.linalg.eigvals(A)` | Autovalores |
| `@(x) x^2` | `lambda x: x**2` | Funcao anonima |
| `1:0.1:10` | `np.arange(1, 10.1, 0.1)` | Vetor com passo |
| `zeros(3,3)` | `np.zeros((3,3))` | Matriz de zeros |
| `A'` | `A.T` | Transposta |
| `inv(A)` | `np.linalg.inv(A)` | Inversa |

---

## Elementos Finitos: Gerando Mesh e Solver

Vamos usar o Claude para construir passo a passo um solver de
elementos finitos 1D para deflexao de uma viga (equacao de Euler-Bernoulli).

**Problema:** Viga engastada-livre (cantilever) com carga distribuida.

Equacao: `EI * d4w/dx4 = q(x)` onde:
- `E` = modulo de Young
- `I` = momento de inercia
- `w` = deflexao
- `q` = carga distribuida

In [ ]:
prompt_fem = '''Escreva um solver de elementos finitos 1D simplificado para
deflexao de uma viga (problema de Poisson 1D como simplificacao).

Problema: -EI * d2w/dx2 = q  (viga simplificada, flexao pura)
Condicoes de contorno: w(0) = 0, w(L) = 0 (bi-apoiada)
Carga: q = 1000 N/m (uniforme)
E*I = 1e6 N.m2
L = 1.0 m

Passos:
1. Gerar mesh 1D com N elementos
2. Montar matrizes de rigidez locais e global
3. Montar vetor de forca
4. Aplicar condicoes de contorno
5. Resolver sistema linear
6. Plotar deflexao e comparar com solucao analitica

Retorne apenas codigo Python com numpy e matplotlib.'''

codigo_fem = ask(prompt_fem, system=system_sci, model=SONNET)
print(codigo_fem)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# FEM 1D - Viga bi-apoiada com carga uniforme
# Equacao: -EI * d2w/dx2 = q
# CC: w(0) = 0, w(L) = 0
# =========================================================

# Parametros fisicos
EI = 1e6       # rigidez flexural (N.m2)
q = 1000.0     # carga distribuida (N/m)
L = 1.0        # comprimento (m)

# Parametros da mesh
N_elem = 20    # numero de elementos
N_nos = N_elem + 1
h = L / N_elem  # tamanho do elemento

# 1. Gerar mesh
x_nos = np.linspace(0, L, N_nos)
print(f'Mesh: {N_elem} elementos, {N_nos} nos, h = {h:.4f} m')

# 2. Matriz de rigidez local (elemento de barra)
# Para -EI*d2w/dx2 = q, a matriz local e:
# K_local = (EI/h) * [[1, -1], [-1, 1]]
K_local = (EI / h) * np.array([[1, -1],
                                [-1, 1]])

# 3. Montagem da matriz global
K_global = np.zeros((N_nos, N_nos))
F_global = np.zeros(N_nos)

for e in range(N_elem):
    # Nos do elemento e
    no_i = e
    no_j = e + 1
    
    # Montar matriz global (assembly)
    K_global[no_i, no_i] += K_local[0, 0]
    K_global[no_i, no_j] += K_local[0, 1]
    K_global[no_j, no_i] += K_local[1, 0]
    K_global[no_j, no_j] += K_local[1, 1]
    
    # Vetor de forca (carga distribuida -> nos)
    f_local = q * h / 2  # metade da carga para cada no
    F_global[no_i] += f_local
    F_global[no_j] += f_local

print(f'Matriz K: {K_global.shape}')
print(f'Vetor F: {F_global.shape}')

# 4. Aplicar condicoes de contorno (w(0) = 0, w(L) = 0)
# Metodo de penalidade: zeramos linhas/colunas dos nos fixos
nos_livres = list(range(1, N_nos - 1))  # nos internos
K_red = K_global[np.ix_(nos_livres, nos_livres)]
F_red = F_global[nos_livres]

# 5. Resolver sistema linear
w_internos = np.linalg.solve(K_red, F_red)

# Montar solucao completa
w = np.zeros(N_nos)
w[1:-1] = w_internos

# 6. Solucao analitica: w(x) = q/(24*EI) * x * (L^3 - 2*L*x^2 + x^3)
x_ana = np.linspace(0, L, 200)
w_ana = q / (24 * EI) * x_ana * (L**3 - 2*L*x_ana**2 + x_ana**3)

# Plotar
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Deflexao
ax1.plot(x_ana, w_ana * 1000, 'b-', lw=2, label='Analitico')
ax1.plot(x_nos, w * 1000, 'ro--', markersize=6, label=f'FEM ({N_elem} elem.)')
ax1.set_xlabel('Posicao x (m)')
ax1.set_ylabel('Deflexao w (mm)')
ax1.set_title('Deflexao da Viga Bi-apoiada')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Erro
w_ana_nos = q / (24 * EI) * x_nos * (L**3 - 2*L*x_nos**2 + x_nos**3)
erro = np.abs(w - w_ana_nos) * 1000  # em mm
ax2.bar(range(N_nos), erro, color='orange', alpha=0.7)
ax2.set_xlabel('Indice do no')
ax2.set_ylabel('Erro absoluto (mm)')
ax2.set_title('Erro FEM vs Analitico')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Metricas
w_max_fem = np.max(w) * 1000
w_max_ana = np.max(w_ana) * 1000
print(f'\nDeflexao maxima (FEM):      {w_max_fem:.6f} mm')
print(f'Deflexao maxima (analitico): {w_max_ana:.6f} mm')
print(f'Erro relativo:               {abs(w_max_fem - w_max_ana)/w_max_ana*100:.4f}%')

### Convergencia do FEM

Vamos verificar como o erro diminui ao refinar a mesh.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def fem_1d_viga(N_elem, EI=1e6, q=1000.0, L=1.0):
    """Solver FEM 1D para viga bi-apoiada."""
    N_nos = N_elem + 1
    h = L / N_elem
    x_nos = np.linspace(0, L, N_nos)
    
    K = np.zeros((N_nos, N_nos))
    F = np.zeros(N_nos)
    Ke = (EI / h) * np.array([[1, -1], [-1, 1]])
    
    for e in range(N_elem):
        K[e:e+2, e:e+2] += Ke
        F[e] += q * h / 2
        F[e+1] += q * h / 2
    
    livres = list(range(1, N_nos - 1))
    w = np.zeros(N_nos)
    w[1:-1] = np.linalg.solve(K[np.ix_(livres, livres)], F[livres])
    return x_nos, w

# Estudo de convergencia
elementos = [4, 8, 16, 32, 64, 128]
erros = []
hs = []

for N in elementos:
    x, w = fem_1d_viga(N)
    h = 1.0 / N
    hs.append(h)
    # Solucao analitica nos nos
    w_ana = 1000 / (24 * 1e6) * x * (1.0**3 - 2*1.0*x**2 + x**3)
    erro_max = np.max(np.abs(w - w_ana))
    erros.append(erro_max)
    print(f'N={N:>4d}  h={h:.4f}  erro_max={erro_max:.2e}')

# Plotar convergencia (log-log)
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(hs, erros, 'bo-', markersize=8, label='Erro FEM')

# Linha de referencia O(h^2)
h_ref = np.array(hs)
ax.loglog(h_ref, erros[0] * (h_ref / h_ref[0])**2, 'r--', label='O(h^2)')

ax.set_xlabel('Tamanho do elemento h (m)')
ax.set_ylabel('Erro maximo (m)')
ax.set_title('Convergencia do FEM 1D - Elementos Lineares')
ax.legend()
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

---

## Exercicios

1. **Monte Carlo**: Modifique a simulacao de decaimento radioativo para incluir
   dois isotopos com meias-vidas diferentes. Plote ambos no mesmo grafico.

2. **Debug**: Escreva um integrador numerico (Euler ou RK4) com 3 bugs
   propositais. Envie ao Claude e veja se ele encontra todos.

3. **Otimizacao**: Pegue o solver da equacao do calor (secao Debug) e
   vetorize o loop interno usando slicing NumPy:
   `u_new[1:-1] = u[1:-1] + r*(u[2:] - 2*u[1:-1] + u[:-2])`
   Compare o tempo com a versao com loop.

4. **Conversao**: Encontre um codigo em Fortran ou MATLAB de um metodo
   numerico (ex: biseccao, Newton-Raphson) e peca ao Claude para converter
   para Python. Valide os resultados.

5. **FEM**: Modifique o solver FEM para usar condicoes de contorno de
   cantilever (engaste em x=0: w(0)=0, dw/dx(0)=0) em vez de bi-apoiada.

In [ ]:
# Seu codigo aqui


---

## Proximos passos

- Explore bibliotecas de FEM em Python: FEniCS, SfePy, scikit-fem
- Use `scipy.integrate.solve_ivp` para ODEs mais complexas
- Teste o Claude com codigos de CFD (dinamica de fluidos computacional)
- Combine Monte Carlo com Machine Learning (ex: redes neurais para PDEs)
- [docs.anthropic.com](https://docs.anthropic.com) - documentacao da API
- [numpy.org/doc](https://numpy.org/doc/stable/) - referencia NumPy
- [scipy.org](https://scipy.org) - SciPy para computacao cientifica